In [1]:
# SPDX-FileCopyrightText: 2024-2026 Dan J. Bower <dbower@eaps.ethz.ch>
# SPDX-FileCopyrightText: 2024-2026 Paolo A. Sossi
#
# SPDX-License-Identifier: Apache-2.0

from pathlib import Path
from typing import Literal

import numpy as np

from atmodeller import ChemicalSpecies, EquilibriumModel, Planet, ReservoirSpecies, console_logger
from atmodeller.eos import get_eos_models
from atmodeller.solubility import get_solubility_models
from atmodeller.thermodata import IronWustiteBuffer

# Log to the console. For more detail, use console_logger(logging.DEBUG)
logger = console_logger()

# Sub-Neptune gas dwarf

This notebook is available at `notebooks/examples/subneptune.ipynb` and is easiest to obtain by downloading the source code.

## Citation

Dan J. Bower, Maggie A. Thompson, Kaustubh Hakim, Meng Tian, and Paolo A. Sossi (2025), Diversity of Low-mass Planet Atmospheres in the C–H–O–N–S–Cl System with Interior Dissolution, Nonideality, and Condensation: Application to TRAPPIST-1e and Sub-Neptunes. The Astrophysical Journal, Volume 995, Number 1, doi: 10.3847/1538-4357/ae1479.

https://iopscience.iop.org/article/10.3847/1538-4357/ae1479

## Initial setup

Each section varies one property of a sub-Neptune with the mass of K2-18b (8.63 Earth masses) and a magma ocean at 3000 K, as in the paper. The model is solved first with ideal gases and then with real-gas equations of state. The real-gas models start from the ideal-gas solution (a warm start), which is a good initial guess, so they converge in a few iterations, whereas the ideal-gas models start from the default initial guess.

The ideal-gas and real-gas models are each built once, and each section updates them with its own values (see Updating a model in the documentation), so the solver of each model is compiled only once, rather than once per section.

The code blocks below must always be run. You can then run only the ideal-gas models, but each real-gas model needs the ideal-gas model of its section to be run first.

In [2]:
# In the paper we perform 5000 simulations
# number_of_realisations = 5000
number_of_realisations = 10

surface_temperature = 3000  # K

# For simulations with fixed mass and surface radius:
planet_mass = 5.154e25
surface_radius = 1.1225e7  # using M-R relation from Hakim+2018
mantle_melt_fraction = 1.0  # 0.1

# Whether to export the data to Excel and pickle files, and where
WRITE_OUTPUT = False
OUTPUT_DIRECTORY = Path("output") / "subneptune"

# There are a few different output formats
output_format: Literal["elements_species", "named_arrays"] = "named_arrays"

if WRITE_OUTPUT:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

Create the gas species, as ideal gases and as real gases with equations of state

In [3]:
H2O_g = ChemicalSpecies.create_gas("H2O")
H2_g = ChemicalSpecies.create_gas("H2")
O2_g = ChemicalSpecies.create_gas("O2")
CO_g = ChemicalSpecies.create_gas("CO")
CO2_g = ChemicalSpecies.create_gas("CO2")
CH4_g = ChemicalSpecies.create_gas("CH4")

ideal_gases = (H2O_g, H2_g, O2_g, CO_g, CO2_g, CH4_g)

In [4]:
eos_models = get_eos_models()

H2O_rg = ChemicalSpecies.create_gas("H2O", activity=eos_models["H2O_cork_holland98"])
H2_rg = ChemicalSpecies.create_gas("H2", activity=eos_models["H2_chabrier21"])
O2_rg = ChemicalSpecies.create_gas("O2")
CO_rg = ChemicalSpecies.create_gas("CO", activity=eos_models["CO_cs_shi92"])
CO2_rg = ChemicalSpecies.create_gas("CO2", activity=eos_models["CO2_cs_shi92"])
CH4_rg = ChemicalSpecies.create_gas("CH4", activity=eos_models["CH4_cs_shi92"])

real_gases = (H2O_rg, H2_rg, O2_rg, CO_rg, CO2_rg, CH4_rg)

Create the melt species

In [5]:
solubility_models = get_solubility_models()

# Exclude mass of species in melt from mass balance constraints, as it is negligible compared to
# the mass of the silicate melt itself.
include_in_phase_mass = False

# Melt species
H2O_d = ReservoirSpecies.create_dissolved(
    "H2O",
    solubility=solubility_models["H2O_basalt_dixon95"],
    include_in_phase_mass=include_in_phase_mass,
)
H2_d = ReservoirSpecies.create_dissolved(
    "H2",
    solubility=solubility_models["H2_basalt_hirschmann12"],
    include_in_phase_mass=include_in_phase_mass,
)
CO_d = ReservoirSpecies.create_dissolved(
    "CO",
    solubility=solubility_models["CO_basalt_yoshioka19"],
    include_in_phase_mass=include_in_phase_mass,
)
CO2_d = ReservoirSpecies.create_dissolved(
    "CO2",
    solubility=solubility_models["CO2_basalt_dixon95"],
    include_in_phase_mass=include_in_phase_mass,
)
CH4_d = ReservoirSpecies.create_dissolved(
    "CH4",
    solubility=solubility_models["CH4_basalt_ardia13"],
    include_in_phase_mass=include_in_phase_mass,
)

melt_species = (H2O_d, H2_d, CO_d, CO2_d, CH4_d)

In [6]:
# One value of the planet mass and surface radius per realisation, so that the sections that vary
# them can update the models below rather than build new ones
sub_neptune_ideal_gases = Planet.from_species(
    ideal_gases,
    silicate_melt_species=melt_species,
    temperature=surface_temperature,
    planet_mass=np.full(number_of_realisations, planet_mass),
    surface_radius=np.full(number_of_realisations, surface_radius),
    mantle_melt_fraction=mantle_melt_fraction,
)

[10:55:04 - atmodeller.phases              - INFO     ] - Creating GasPhase: ('H2O_g: IdealGas', 'H2_g: IdealGas', 'O2_g: IdealGas', 'CO_g: IdealGas', 'CO2_g: IdealGas', 'CH4_g: IdealGas')


[10:55:04 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ('H2O_d: SolubilityPowerLaw', 'H2_d: SolubilityPowerLawLog10', 'CO_d: _CO_basalt_yoshioka19', 'CO2_d: _CO2_basalt_dixon95', 'CH4_d: _CH4_basalt_ardia13')


[10:55:04 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


[10:55:04 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


[10:55:04 - atmodeller.reactions           - INFO     ] - ReactionNetwork network = {0: '1.0 H2_g + 0.5 O2_g = 1.0 H2O_g',
 1: '0.5 O2_g + 1.0 CO_g = 1.0 CO2_g',
 2: '2.0 H2_g + 1.0 CO_g = 0.5 O2_g + 1.0 CH4_g'}


[10:55:04 - atmodeller.reactions           - INFO     ] - Thermodynamic data of all species cover 200 K to 6000 K, outside which each species follows its extrapolation


[10:55:04 - atmodeller.reactions           - INFO     ] - DissolutionNetwork network = {0: '1.0 H2O_g = 1.0 H2O_d',
 1: '1.0 H2_g = 1.0 H2_d',
 2: '1.0 CO_g = 1.0 CO_d',
 3: '1.0 CO2_g = 1.0 CO2_d',
 4: '1.0 CH4_g = 1.0 CH4_d'}


[10:55:04 - atmodeller.reactions           - INFO     ] - ReactionSystem network = {0: '1.0 H2_g + 0.5 O2_g = 1.0 H2O_g',
 1: '0.5 O2_g + 1.0 CO_g = 1.0 CO2_g',
 2: '2.0 H2_g + 1.0 CO_g = 0.5 O2_g + 1.0 CH4_g',
 3: '1.0 H2O_g = 1.0 H2O_d',
 4: '1.0 H2_g = 1.0 H2_d',
 5: '1.0 CO_g = 1.0 CO_d',
 6: '1.0 CO2_g = 1.0 CO2_d',
 7: '1.0 CH4_g = 1.0 CH4_d'}


In [7]:
sub_neptune_real_gases = Planet.from_species(
    real_gases,
    silicate_melt_species=melt_species,
    temperature=surface_temperature,
    planet_mass=np.full(number_of_realisations, planet_mass),
    surface_radius=np.full(number_of_realisations, surface_radius),
    mantle_melt_fraction=mantle_melt_fraction,
)

# Each model is built once and updated in each section below, so that its solver is compiled only
# once. The constraints are set in each section, so they are inactive (NaN) here.
no_constraint = np.full(number_of_realisations, np.nan)
ideal_model = EquilibriumModel.from_state(
    sub_neptune_ideal_gases,
    mass_constraints={"H": no_constraint, "C": no_constraint},
    activity_constraints={"O2_g": IronWustiteBuffer(no_constraint)},
)
real_model = EquilibriumModel.from_state(
    sub_neptune_real_gases,
    mass_constraints={"H": no_constraint, "C": no_constraint},
    activity_constraints={"O2_g": IronWustiteBuffer(no_constraint)},
)

[10:55:04 - atmodeller.phases              - INFO     ] - Creating GasPhase: ('H2O_g: CombinedRealGas', 'H2_g: CombinedRealGas', 'O2_g: IdealGas', 'CO_g: CombinedRealGas', 'CO2_g: CombinedRealGas', 'CH4_g: CombinedRealGas')


[10:55:04 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ('H2O_d: SolubilityPowerLaw', 'H2_d: SolubilityPowerLawLog10', 'CO_d: _CO_basalt_yoshioka19', 'CO2_d: _CO2_basalt_dixon95', 'CH4_d: _CH4_basalt_ardia13')


[10:55:04 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


[10:55:04 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


[10:55:04 - atmodeller.reactions           - INFO     ] - ReactionNetwork network = {0: '1.0 H2_g + 0.5 O2_g = 1.0 H2O_g',
 1: '0.5 O2_g + 1.0 CO_g = 1.0 CO2_g',
 2: '2.0 H2_g + 1.0 CO_g = 0.5 O2_g + 1.0 CH4_g'}


[10:55:04 - atmodeller.reactions           - INFO     ] - Thermodynamic data of all species cover 200 K to 6000 K, outside which each species follows its extrapolation


[10:55:04 - atmodeller.reactions           - INFO     ] - DissolutionNetwork network = {0: '1.0 H2O_g = 1.0 H2O_d',
 1: '1.0 H2_g = 1.0 H2_d',
 2: '1.0 CO_g = 1.0 CO_d',
 3: '1.0 CO2_g = 1.0 CO2_d',
 4: '1.0 CH4_g = 1.0 CH4_d'}


[10:55:04 - atmodeller.reactions           - INFO     ] - ReactionSystem network = {0: '1.0 H2_g + 0.5 O2_g = 1.0 H2O_g',
 1: '0.5 O2_g + 1.0 CO_g = 1.0 CO2_g',
 2: '2.0 H2_g + 1.0 CO_g = 0.5 O2_g + 1.0 CH4_g',
 3: '1.0 H2O_g = 1.0 H2O_d',
 4: '1.0 H2_g = 1.0 H2_d',
 5: '1.0 CO_g = 1.0 CO_d',
 6: '1.0 CO2_g = 1.0 CO2_d',
 7: '1.0 CH4_g = 1.0 CH4_d'}


## Vary Hydrogen Mass Fraction

The hydrogen mass varies from 0.1 to 3% of the planet mass, with the C/H mass ratio at 100 times solar and the oxygen fugacity at IW-3.

In [8]:
log10_H_frac = np.linspace(-1.0, 0.5, number_of_realisations)  # 0.1 to 3% of planet mass
log10_ch_ratios = np.full(number_of_realisations, -0.5)  # 100X Solar
fO2_log10_shifts = np.full(number_of_realisations, -3)

h_kg = ((10**log10_H_frac) / 100) * planet_mass
c_kg = h_kg * 10**log10_ch_ratios

mass_constraints = {"H": h_kg, "C": c_kg}

fugacity_constraints = {"O2_g": IronWustiteBuffer(fO2_log10_shifts)}

### Ideal gas with solubilities

In [9]:
ideal_withsol = ideal_model.update_constraints(
    mass_constraints=mass_constraints, activity_constraints=fugacity_constraints
)

output_ideal_withsol = ideal_withsol.solve_with_default()

output_ideal_withsol.solver_stats_to_logger()

# Quick look at the solution
# output_ideal_withsol.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_ideal_withsol.to_excel(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_ideal_with_solubility",
        output_format=output_format,
    )

    # Write the data to a pickle file with dataframes
    output_ideal_withsol.to_pickle(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_ideal_with_solubility",
        output_format=output_format,
    )

[10:55:04 - atmodeller.classes             - INFO     ] - Initial guess: 10 model(s) from ElementPotentialInitializer, 0 model(s) from the given initial guess


[10:55:08 - atmodeller.output              - INFO     ] - Solve complete: 10 (100.00%) successful model(s)


[10:55:08 - atmodeller.output              - INFO     ] - Solve attempt 1: 10 (100.00%) model(s) solved


[10:55:08 - atmodeller.output              - INFO     ] - Solver steps (max) = 1


### Real gas with solubility

In [10]:
real_withsol = real_model.update_constraints(
    mass_constraints=mass_constraints, activity_constraints=fugacity_constraints
)

output_real_withsol = real_withsol.solve(output_ideal_withsol.solution)

output_real_withsol.solver_stats_to_logger()

# Quick look at the solution
# output_real_withsol.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_real_withsol.to_excel(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_real_with_solubility",
        output_format=output_format,
    )

    # Write the data to a pickle file with dataframes
    output_real_withsol.to_pickle(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_real_with_solubility",
        output_format=output_format,
    )

[10:55:08 - atmodeller.classes             - INFO     ] - Initial guess: 0 model(s) from ElementPotentialInitializer, 10 model(s) from the given initial guess


[10:55:34 - atmodeller.output              - INFO     ] - Solve complete: 10 (100.00%) successful model(s)


[10:55:34 - atmodeller.output              - INFO     ] - Solve attempt 1: 10 (100.00%) model(s) solved


[10:55:34 - atmodeller.output              - INFO     ] - Solver steps (max) = 6


## Vary Oxygen Fugacity

The oxygen fugacity varies from IW-6 to IW, with hydrogen at 1% of the planet mass and the C/H mass ratio at 100 times solar.

In [11]:
fO2_log10_shifts = np.linspace(-6, 0, number_of_realisations)  # IW-6 to IW
log10_H_frac = np.full(number_of_realisations, 0)  # 1% of planet mass
log10_ch_ratios = np.full(number_of_realisations, -0.5)  # 100X Solar

h_kg = ((10**log10_H_frac) / 100) * planet_mass
c_kg = h_kg * 10**log10_ch_ratios

mass_constraints = {"H": h_kg, "C": c_kg}

fugacity_constraints = {"O2_g": IronWustiteBuffer(fO2_log10_shifts)}

### Ideal gas with solubility

In [12]:
ideal_withsol_varyfO2 = ideal_model.update_constraints(
    mass_constraints=mass_constraints, activity_constraints=fugacity_constraints
)

output_ideal_withsol_varyfO2 = ideal_withsol_varyfO2.solve_with_default()

output_ideal_withsol_varyfO2.solver_stats_to_logger()

# Quick look at the solution
# output_ideal_withsol_varyfO2.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_ideal_withsol_varyfO2.to_excel(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_ideal_with_solubility_varyfO2",
        output_format=output_format,
    )

    # Write the data to a pickle file with dataframes
    output_ideal_withsol_varyfO2.to_pickle(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_ideal_with_solubility_varyfO2",
        output_format=output_format,
    )

[10:55:34 - atmodeller.classes             - INFO     ] - Initial guess: 10 model(s) from ElementPotentialInitializer, 0 model(s) from the given initial guess


[10:55:34 - atmodeller.output              - INFO     ] - Solve complete: 10 (100.00%) successful model(s)


[10:55:34 - atmodeller.output              - INFO     ] - Solve attempt 1: 10 (100.00%) model(s) solved


[10:55:34 - atmodeller.output              - INFO     ] - Solver steps (max) = 1


### Real gas with solubility

In [13]:
real_withsol_varyfO2 = real_model.update_constraints(
    mass_constraints=mass_constraints, activity_constraints=fugacity_constraints
)

output_real_withsol_varyfO2 = real_withsol_varyfO2.solve(output_ideal_withsol_varyfO2.solution)

output_real_withsol_varyfO2.solver_stats_to_logger()

# Quick look at the solution
# output_real_withsol_varyfO2.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_real_withsol_varyfO2.to_excel(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_real_with_solubility_varyfO2",
        output_format=output_format,
    )

    # Write the data to a pickle file with dataframes
    output_real_withsol_varyfO2.to_pickle(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_real_with_solubility_varyfO2",
        output_format=output_format,
    )

[10:55:34 - atmodeller.classes             - INFO     ] - Initial guess: 0 model(s) from ElementPotentialInitializer, 10 model(s) from the given initial guess


[10:55:34 - atmodeller.output              - INFO     ] - Solve complete: 10 (100.00%) successful model(s)


[10:55:34 - atmodeller.output              - INFO     ] - Solve attempt 1: 10 (100.00%) model(s) solved


[10:55:34 - atmodeller.output              - INFO     ] - Solver steps (max) = 6


## Vary C/H Ratio

The C/H mass ratio varies from solar (log10(C/H) = -2.5) to bulk silicate Earth (log10(C/H) = 0.1), with hydrogen at 1% of the planet mass and the oxygen fugacity at IW-3.

In [14]:
log10_ch_ratios = np.linspace(-2.5, 0.1, number_of_realisations)  # Solar to BSE Ratios
log10_H_frac = np.full(number_of_realisations, 0)  # 1% of planet mass
fO2_log10_shifts = np.full(number_of_realisations, -3)

h_kg = ((10**log10_H_frac) / 100) * planet_mass
c_kg = h_kg * 10**log10_ch_ratios

mass_constraints = {"H": h_kg, "C": c_kg}
fugacity_constraints = {"O2_g": IronWustiteBuffer(fO2_log10_shifts)}

### Ideal gas with solubility

In [15]:
ideal_withsol_varyCtoH = ideal_model.update_constraints(
    mass_constraints=mass_constraints, activity_constraints=fugacity_constraints
)

output_ideal_withsol_varyCtoH = ideal_withsol_varyCtoH.solve_with_default()

output_ideal_withsol_varyCtoH.solver_stats_to_logger()

# Quick look at the solution
# output_ideal_withsol_varyCtoH.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_ideal_withsol_varyCtoH.to_excel(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_ideal_with_solubility_varyCtoH",
        output_format=output_format,
    )

    # Write the data to a pickle file with dataframes
    output_ideal_withsol_varyCtoH.to_pickle(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_ideal_with_solubility_varyCtoH",
        output_format=output_format,
    )

[10:55:34 - atmodeller.classes             - INFO     ] - Initial guess: 10 model(s) from ElementPotentialInitializer, 0 model(s) from the given initial guess


[10:55:34 - atmodeller.output              - INFO     ] - Solve complete: 10 (100.00%) successful model(s)


[10:55:34 - atmodeller.output              - INFO     ] - Solve attempt 1: 10 (100.00%) model(s) solved


[10:55:34 - atmodeller.output              - INFO     ] - Solver steps (max) = 1


### Real gas with solubility

In [16]:
real_withsol_varyCtoH = real_model.update_constraints(
    mass_constraints=mass_constraints, activity_constraints=fugacity_constraints
)

output_real_withsol_varyCtoH = real_withsol_varyCtoH.solve(output_ideal_withsol_varyCtoH.solution)

output_real_withsol_varyCtoH.solver_stats_to_logger()

# Quick look at the solution
# output_real_withsol_varyCtoH.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_real_withsol_varyCtoH.to_excel(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_real_with_solubility_varyCtoH",
        output_format=output_format,
    )

    # Write the data to a pickle file with dataframes
    output_real_withsol_varyCtoH.to_pickle(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_real_with_solubility_varyCtoH",
        output_format=output_format,
    )

[10:55:34 - atmodeller.classes             - INFO     ] - Initial guess: 0 model(s) from ElementPotentialInitializer, 10 model(s) from the given initial guess


[10:55:34 - atmodeller.output              - INFO     ] - Solve complete: 10 (100.00%) successful model(s)


[10:55:34 - atmodeller.output              - INFO     ] - Solve attempt 1: 10 (100.00%) model(s) solved


[10:55:34 - atmodeller.output              - INFO     ] - Solver steps (max) = 6


## Vary Planetary Surface Radius

The surface radius varies from 1.76 to 2.6 Earth radii, with the planet mass fixed at 8.63 Earth masses, hydrogen at about 3% of the planet mass, the C/H mass ratio at 100 times solar, and the oxygen fugacity at IW-3.

In [17]:
surface_radius = np.linspace(1.1225e7, 1.6647e7, number_of_realisations)  # Vary linearly
log10_H_frac = np.full(
    number_of_realisations, 0.5
)  # ~3% of planet mass, used for fix Surf Radius and Planet Mass cases
log10_ch_ratios = np.full(number_of_realisations, -0.5)  # 100X Solar
fO2_log10_shifts = np.full(number_of_realisations, -3)

h_kg = ((10**log10_H_frac) / 100) * planet_mass
c_kg = h_kg * 10**log10_ch_ratios

mass_constraints = {"H": h_kg, "C": c_kg}

fugacity_constraints = {"O2_g": IronWustiteBuffer(fO2_log10_shifts)}

### Ideal gas with solubility

In [18]:
ideal_withsol_varyRsurf = ideal_model.update_state(
    surface_radius=surface_radius
).update_constraints(mass_constraints=mass_constraints, activity_constraints=fugacity_constraints)

output_ideal_withsol_varyRsurf = ideal_withsol_varyRsurf.solve_with_default()

output_ideal_withsol_varyRsurf.solver_stats_to_logger()

# Quick look at the solution
# output_ideal_withsol_varyRsurf.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_ideal_withsol_varyRsurf.to_excel(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_ideal_with_solubility_varyRsurf",
        output_format=output_format,
    )

    # Write the data to a pickle file with dataframes
    output_ideal_withsol_varyRsurf.to_pickle(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_ideal_with_solubility_varyRsurf",
        output_format=output_format,
    )

[10:55:34 - atmodeller.classes             - INFO     ] - Initial guess: 10 model(s) from ElementPotentialInitializer, 0 model(s) from the given initial guess


[10:55:34 - atmodeller.output              - INFO     ] - Solve complete: 10 (100.00%) successful model(s)


[10:55:34 - atmodeller.output              - INFO     ] - Solve attempt 1: 10 (100.00%) model(s) solved


[10:55:34 - atmodeller.output              - INFO     ] - Solver steps (max) = 1


### Real gas with solubility

In [19]:
real_withsol_varyRsurf = real_model.update_state(surface_radius=surface_radius).update_constraints(
    mass_constraints=mass_constraints, activity_constraints=fugacity_constraints
)

output_real_withsol_varyRsurf = real_withsol_varyRsurf.solve(
    output_ideal_withsol_varyRsurf.solution
)

output_real_withsol_varyRsurf.solver_stats_to_logger()

# Quick look at the solution
# output_real_withsol_varyRsurf.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_real_withsol_varyRsurf.to_excel(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_real_with_solubility_varyRsurf",
        output_format=output_format,
    )

    # Write the data to a pickle file with dataframes
    output_real_withsol_varyRsurf.to_pickle(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_real_with_solubility_varyRsurf",
        output_format=output_format,
    )

[10:55:34 - atmodeller.classes             - INFO     ] - Initial guess: 0 model(s) from ElementPotentialInitializer, 10 model(s) from the given initial guess


[10:55:34 - atmodeller.output              - INFO     ] - Solve complete: 10 (100.00%) successful model(s)


[10:55:34 - atmodeller.output              - INFO     ] - Solve attempt 1: 10 (100.00%) model(s) solved


[10:55:34 - atmodeller.output              - INFO     ] - Solver steps (max) = 6


## Vary Planetary Mass

The planet mass varies from 4 to 9 Earth masses, with the surface radius fixed at 1.76 Earth radii, hydrogen at about 3% of the planet mass, the C/H mass ratio at 100 times solar, and the oxygen fugacity at IW-3.

In [20]:
surface_radius = 1.1225e7  # using M-R relation from Hakim+2018

# For simulations with varying planet mass:
planet_mass_Earths = np.linspace(4, 9, number_of_realisations)  # Vary linearly from 4-9 MEarth
planet_mass = planet_mass_Earths * 5.9722e24

log10_H_frac = np.full(
    number_of_realisations, 0.5
)  # ~3% of planet mass, used for fix Surf Radius and Planet Mass cases
log10_ch_ratios = np.full(number_of_realisations, -0.5)  # 100X Solar
fO2_log10_shifts = np.full(number_of_realisations, -3)

h_kg = ((10**log10_H_frac) / 100) * planet_mass
c_kg = h_kg * 10**log10_ch_ratios

mass_constraints = {"H": h_kg, "C": c_kg}

fugacity_constraints = {"O2_g": IronWustiteBuffer(fO2_log10_shifts)}

### Ideal gas with solubility

In [21]:
ideal_withsol_varyMp = ideal_model.update_state(
    planet_mass=planet_mass, surface_radius=surface_radius
).update_constraints(mass_constraints=mass_constraints, activity_constraints=fugacity_constraints)

output_ideal_withsol_varyMp = ideal_withsol_varyMp.solve_with_default()

output_ideal_withsol_varyMp.solver_stats_to_logger()

# Quick look at the solution
# output_ideal_withsol_varyMp.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_ideal_withsol_varyMp.to_excel(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_ideal_with_solubility_varyMp",
        output_format=output_format,
    )

    # Write the data to a pickle file with dataframes
    output_ideal_withsol_varyMp.to_pickle(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_ideal_with_solubility_varyMp",
        output_format=output_format,
    )

[10:55:34 - atmodeller.classes             - INFO     ] - Initial guess: 10 model(s) from ElementPotentialInitializer, 0 model(s) from the given initial guess


[10:55:34 - atmodeller.output              - INFO     ] - Solve complete: 10 (100.00%) successful model(s)


[10:55:34 - atmodeller.output              - INFO     ] - Solve attempt 1: 10 (100.00%) model(s) solved


[10:55:34 - atmodeller.output              - INFO     ] - Solver steps (max) = 1


### Real gas with solubility

In [22]:
real_withsol_varyMp = real_model.update_state(
    planet_mass=planet_mass, surface_radius=surface_radius
).update_constraints(mass_constraints=mass_constraints, activity_constraints=fugacity_constraints)

output_real_withsol_varyMp = real_withsol_varyMp.solve(output_ideal_withsol_varyMp.solution)

output_real_withsol_varyMp.solver_stats_to_logger()

# Quick look at the solution
# output_real_withsol_varyMp.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_real_withsol_varyMp.to_excel(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_real_with_solubility_varyMp",
        output_format=output_format,
    )

    # Write the data to a pickle file with dataframes
    output_real_withsol_varyMp.to_pickle(
        OUTPUT_DIRECTORY / f"sub_neptune_{surface_temperature}K_real_with_solubility_varyMp",
        output_format=output_format,
    )

[10:55:34 - atmodeller.classes             - INFO     ] - Initial guess: 0 model(s) from ElementPotentialInitializer, 10 model(s) from the given initial guess


[10:55:34 - atmodeller.output              - INFO     ] - Solve complete: 10 (100.00%) successful model(s)


[10:55:34 - atmodeller.output              - INFO     ] - Solve attempt 1: 10 (100.00%) model(s) solved


[10:55:34 - atmodeller.output              - INFO     ] - Solver steps (max) = 6
